# AI Travel Operations Agent

## Project Introduction

The AI Travel Operations Agent is an intelligent multi-agent system designed to automate and simplify travel planning.

The system uses Large Language Models (LLMs), LangGraph, and the Model Context Protocol (MCP) to coordinate specialized AI agents.

Each agent performs a specific task, such as transportation research, hotel discovery, weather analysis, activity recommendations, budget estimation, and itinerary verification.

### Project Objectives

1. Understand user travel requirements.
2. Discover suitable transportation options.
3. Find hotels based on budget and preferences.
4. Analyze weather conditions.
5. Recommend tourist attractions and activities.
6. Estimate total travel expenses.
7. Generate a personalized travel itinerary.
8. Verify information using connected data sources.

### Example User Request

"Plan a 3-day trip from Hyderabad to Goa for 2 travelers with a total budget of INR 15,000."

### Expected Output

- Transportation recommendations
- Hotel recommendations
- Weather information
- Tourist attractions
- Day-wise travel itinerary
- Estimated total budget

## Technology Stack

| Technology | Purpose |
|---|---|
| Python | Core development language |
| Jupyter Notebook | Development and experimentation |
| Groq API | LLM inference |
| LangChain | LLM integration |
| LangGraph | Multi-agent workflow orchestration |
| MCP | Standardized tool integration |
| Pydantic | Data validation |
| HTTPX | External API requests |
| FastAPI | Backend API development |
| PostgreSQL | Persistent data storage |
| React | Frontend development |
| Tailwind CSS | User interface styling |

## System Architecture

User Travel Request
        |
        v
Travel Request Validation
        |
        v
LangGraph Orchestrator
        |
        +---- Transport Agent
        |
        +---- Hotel Agent
        |
        +---- Weather Agent
        |
        +---- Activities Agent
        |
        +---- Budget Agent
        |
        +---- Review Agent
        |
        v
External APIs and MCP Tools
        |
        v
Final Travel Itinerary

# Import Required Libraries

In [9]:
import os
import sys
import platform

from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [10]:
print("Project: AI Travel Operations Agent")
print("Python Version:", sys.version.split()[0])
print("Operating System:", platform.system())
print("Working Directory:", Path.cwd())

Project: AI Travel Operations Agent
Python Version: 3.12.4
Operating System: Windows
Working Directory: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent\notebooks


## Environment Configuration

Environment variables are used to store sensitive configuration values, such as API keys.

The GROQ_API_KEY is stored in a .env file instead of being hardcoded in the source code.

In [11]:
PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing.")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [12]:
MODEL_NAME = "openai/gpt-oss-120b"

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized successfully.")
print("Selected Model:", MODEL_NAME)

Groq client initialized successfully.
Selected Model: openai/gpt-oss-120b


# Test LLM Response

In [13]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an intelligent AI Travel Assistant. "
                "Provide clear, practical, and structured travel guidance."
            )
        },
        {
            "role": "user",
            "content": (
                "Introduce yourself and explain how you can "
                "assist users with travel planning."
            )
        }
    ],
    temperature=0.3,
    max_completion_tokens=300
)

answer = response.choices[0].message.content

print("AI Travel Assistant Response:\n")
print(answer)

AI Travel Assistant Response:

**Hello! I’m your AI Travel Assistant – a friendly, knowledgeable companion that helps you plan smooth, enjoyable trips from start to finish.**  

Below is a quick overview of what I can do for you, organized into the main stages of travel planning:

---

## 1️⃣ Destination Discovery  
- **Idea generation:** Suggest destinations based on interests (culture, adventure, food, relaxation, wildlife, etc.).  
- **Best‑time advice:** Highlight optimal travel windows, weather patterns, and local events or festivals.  
- **Travel restrictions:** Provide up‑to‑date visa, vaccination, and entry‑requirement information.

## 2️⃣ Budget & Timeline Planning  
- **Cost estimates:** Break down typical expenses for flights, accommodation, meals, transport, and activities.  
- **Budget tools:** Help you set daily spending limits and suggest ways to stretch your money.  
- **Trip length:** Recommend ideal trip durations for different destinations and travel styles.

## 3️⃣ 

# Create a Reusable LLM Function

In [14]:
def generate_llm_response(
    user_prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant."
) -> str:

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.3,
        max_completion_tokens=500
    )

    return response.choices[0].message.content or ""

In [15]:
result = generate_llm_response(
    user_prompt="Suggest 5 popular tourist attractions in Goa."
)

print(result)

Here are five of the most popular tourist attractions in Goa, each offering a different flavor of the state’s vibrant culture, natural beauty, and history:

| # | Attraction | Why It’s Popular | Quick Tips |
|---|------------|------------------|------------|
| **1** | **Baga & Calangute Beaches** | The iconic “Party Beaches” of North Goa. Golden sand, lively shacks, water‑sports (parasailing, jet‑skiing), and a buzzing nightlife scene. | Arrive early to snag a good spot on the sand; sunset at Baga is spectacular. Dress modestly when moving away from the beach area. |
| **2** | **Fort Aguada & Sinquerim Beach** | A well‑preserved 17th‑century Portuguese fort with a lighthouse offering panoramic sea views. The adjoining Sinquerim Beach is quieter than Baga, perfect for swimming. | Bring a hat and water; the fort’s steps can be steep. Entry is free, but a small parking fee applies if you drive. |
| **3** | **Old Goa (Basilica of Bom Jesus & Se Cathedral)** | A UNESCO World Heritage site s

# Create the Travel Request Schema

In [17]:
# Import Pydantic
from pydantic import BaseModel, Field, model_validator
from datetime import date
from typing import Optional

In [18]:
# Define the Schema

class TravelRequest(BaseModel):
    source: str = Field(
        ...,
        min_length=2,
        description="Starting city of the trip"
    )

    destination: str = Field(
        ...,
        min_length=2,
        description="Destination city"
    )

    start_date: date = Field(
        ...,
        description="Trip start date"
    )

    end_date: date = Field(
        ...,
        description="Trip end date"
    )

    travelers: int = Field(
        default=1,
        ge=1,
        le=20,
        description="Number of travelers"
    )

    budget: float = Field(
        ...,
        gt=0,
        description="Total trip budget in INR"
    )

    currency: str = "INR"

    preferences: list[str] = Field(
        default_factory=list,
        description="User travel preferences"
    )

    @model_validator(mode="after")
    def validate_trip(self):
        if self.end_date < self.start_date:
            raise ValueError(
                "End date must be on or after start date."
            )

        if self.source.strip().casefold() == self.destination.strip().casefold():
            raise ValueError(
                "Source and destination must be different."
            )

        return self

In [19]:
# Test the Schema

travel_request = TravelRequest(
    source="Hyderabad",
    destination="Goa",
    start_date="2026-11-10",
    end_date="2026-11-12",
    travelers=2,
    budget=15000,
    preferences=["Beaches", "Local Food", "Sightseeing"]
)

print(travel_request)

source='Hyderabad' destination='Goa' start_date=datetime.date(2026, 11, 10) end_date=datetime.date(2026, 11, 12) travelers=2 budget=15000.0 currency='INR' preferences=['Beaches', 'Local Food', 'Sightseeing']


In [20]:
travel_data = travel_request.model_dump(mode="json")

print(travel_data)

{'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}


In [21]:
# Create the Shared Agent State

from typing import TypedDict, Optional, Any

class TravelAgentState(TypedDict, total=False):
    travel_request: dict[str, Any]

    transport_options: list[dict[str, Any]]
    hotel_options: list[dict[str, Any]]
    weather_info: dict[str, Any]
    activities: list[dict[str, Any]]

    budget_summary: dict[str, Any]
    review_result: dict[str, Any]

    itinerary: dict[str, Any]
    errors: list[str]

In [22]:
# Initialize the Shared State

initial_state: TravelAgentState = {
    "travel_request": travel_data,
    "transport_options": [],
    "hotel_options": [],
    "weather_info": {},
    "activities": [],
    "budget_summary": {},
    "review_result": {},
    "itinerary": {},
    "errors": []
}

print("Initial Travel Agent State:")
print(initial_state)

Initial Travel Agent State:
{'travel_request': {'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}, 'transport_options': [], 'hotel_options': [], 'weather_info': {}, 'activities': [], 'budget_summary': {}, 'review_result': {}, 'itinerary': {}, 'errors': []}


In [24]:
# Validate Incorrect Input
# Test Validation

try:
    invalid_request = TravelRequest(
        source="Hyderabad",
        destination="Goa",
        start_date="2026-11-12",
        end_date="2026-11-10",
        travelers=2,
        budget=15000
    )
except Exception as error:
    print("Validation Error:")
    print(error)

Validation Error:
1 validation error for TravelRequest
  Value error, End date must be on or after start date. [type=value_error, input_value={'source': 'Hyderabad', '...rs': 2, 'budget': 15000}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


# Groq LLM Integration & Reusable LLM Service

In [25]:
import os
from groq import Groq
from dotenv import load_dotenv
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing from .env")

client = Groq(api_key=GROQ_API_KEY)

print("Groq client ready!")
print("Model:", MODEL_NAME)

Groq client ready!
Model: openai/gpt-oss-120b
